## Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR


## Dataset import

We must use the unprocessed data, because the preprocessing done before is not useful for this regression task


In [2]:
df_test = pd.read_csv('/home/tsalvador/Documents/Polito/ML4N-PoliTo/Datasets/https_test.csv')
df_train = pd.read_csv('/home/tsalvador/Documents/Polito/ML4N-PoliTo/Datasets/https_training.csv')


for i in df_train.columns: #reromve non numeric columns
    if not(i.startswith('_')):
        df_train.drop(i, axis=1, inplace=True) 


for i in df_test.columns: #reromve non numeric columns
    if not(i.startswith('_')):
        df_test.drop(i, axis=1, inplace=True) 


In [3]:
def regression_metrics(y_true, y_pred, name):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


Data preparation (s_bytes_all)

In [4]:

# Target: s_bytes_all
ytr_bytes = df_train["_s_bytes_all"]
yte_bytes = df_test["_s_bytes_all"]

# Remove forbidden column
Xtr_bytes = df_train.drop(columns=["_s_bytes_all", "_s_bytes_uniq"])
Xte_bytes = df_test.drop(columns=["_s_bytes_all", "_s_bytes_uniq"])


Data preparation (s_rtt_avg)

In [5]:
# Remove zero RTT values ONLY ON TRAINING SET
dft = df_train[df_train["_s_rtt_avg"] != 0].copy()

ytr_rtt = dft["_s_rtt_avg"]
yte_rtt = df_test["_s_rtt_avg"]


# Remove all RTT-related columns
Xtr_rtt = dft.drop(
    columns=[c for c in dft.columns if "rtt" in c.lower()]
)

Xte_rtt = df_test.drop(
    columns=[c for c in df_test.columns if "rtt" in c.lower()]
)


Cell 5 — Define models

In [6]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor()
}


Cell 6 — Train & evaluate (default hyperparameters)

In [ ]:
results_default = []

for name, model in models.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    # ----- s_bytes_all -----
    pipe.fit(Xtr_bytes, ytr_bytes)
    results_default.append(
        regression_metrics(ytr_bytes, pipe.predict(Xtr_bytes), f"{name}_bytes_train")
    )
    results_default.append(
        regression_metrics(yte_bytes, pipe.predict(Xte_bytes), f"{name}_bytes_test")
    )

    # ----- s_rtt_avg -----
    pipe.fit(Xtr_rtt, ytr_rtt)
    results_default.append(
        regression_metrics(ytr_rtt, pipe.predict(Xtr_rtt), f"{name}_rtt_train")
    )
    results_default.append(
        regression_metrics(yte_rtt, pipe.predict(Xte_rtt), f"{name}_rtt_test")
    )

results_default = pd.DataFrame(results_default)
results_default


Cell 7 — Hyperparameter grids

In [ ]:
param_grids = {
    "LinearRegression": {},

    "RandomForest": {
        "model__n_estimators": [100, 300],
        "model__max_depth": [None, 10, 30]
    },

    "SVR": {
        "model__C": [1, 10],
        "model__gamma": ["scale", "auto"]
    }
}


Cell 8 — Hyperparameter tuning (cross-validation)

In [ ]:
results_tuned = []

for name, model in models.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    grid = GridSearchCV(
        pipe,
        param_grids[name],
        scoring="neg_mean_absolute_error",
        cv=5,
        n_jobs=-1
    )

    # ----- s_bytes_all -----
    grid.fit(Xb_train, yb_train)
    best = grid.best_estimator_
    results_tuned.append(
        regression_metrics(yb_test, best.predict(Xb_test), f"{name}_bytes_tuned")
    )

    # ----- s_rtt_avg -----
    grid.fit(Xr_train, yr_train)
    best = grid.best_estimator_
    results_tuned.append(
        regression_metrics(yr_test, best.predict(Xr_test), f"{name}_rtt_tuned")
    )

results_tuned = pd.DataFrame(results_tuned)
results_tuned


Cell 9 — Final MAE comparison   

In [ ]:
results_tuned[["model", "MAE"]]
